# A08 - Perceptrón multicapa (MLP con optimización de hiperparámetros(Neuronas))

### Nombre: Claudio Alejo Encarnación Martínez
### Fecha: 19 octubre 2026

### Importación de librerías

In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier

In [2]:
df = pd.read_csv("/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/Heart Prediction Quantum Dataset.csv")
df.head()

,Age,Gender,BloodPressure,Cholesterol,HeartRate,HeartDisease
0,68,Hombre,105.00,191,107,1
1,58,Mujer,97.00,249,89,0
2,44,Mujer,93.00,190,82,1
3,72,Hombre,93.00,183,101,1
4,37,Mujer,145.00,166,103,1


### Limpieza de datos

In [3]:
df['BloodPressure'] = df['BloodPressure'].str.replace(',', '.').astype(float)
df['BloodPressure'].head()

df['Gender'] = df['Gender'].str.lower()
df['Gender'] = df['Gender'].str.rstrip("   ")
df['Gender'] = df['Gender'].str.rstrip("}",).str.rstrip("{")
df['Gender'] = df['Gender'].replace('ombre', 'hombre')
df['Gender'] = df['Gender'].map({'hombre': 1, 'mujer': 0})

df = df.drop_duplicates()
df = df.dropna()

df.head()

,Age,Gender,BloodPressure,Cholesterol,HeartRate,HeartDisease
0,68,1,105.0,191,107,1
1,58,0,97.0,249,89,0
2,44,0,93.0,190,82,1
3,72,1,93.0,183,101,1
4,37,0,145.0,166,103,1


### Definición de variables predictoras y variable objetivo

In [4]:
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

### Definición del modelo

In [5]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(max_iter=1000, random_state=42)

### Training and Test Split

In [6]:
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

param_grid = {'hidden_layer_sizes': [
        (c1, c2, c3)
        for c1 in range(5, 16)     
        for c2 in range(10, 21)    
        for c3 in range(5, 11)     
    ]
}

grid = GridSearchCV(model, param_grid, cv=kf)

grid.fit(X_train, y_train)
print(grid.best_params_)

/opt/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/neural_

{'hidden_layer_sizes': (11, 10, 5)}


In [ ]:
from sklearn.preprocessing import StandardScaler

# 2. Pipeline: escalado + MLP
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(random_state=42))
])

# 3. Espacio de búsqueda
param_grid = {
    'mlp__hidden_layer_sizes': [
        (c1, c2, c3)
        for c1 in range(5, 16)
        for c2 in range(10, 21)
        for c3 in range(5, 11)
    ]
}
print(f"Combinaciones: {len(param_grid['mlp__hidden_layer_sizes'])}")

# 4. K-Fold (lo usa GridSearchCV internamente, NO lo iteres a mano)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 5. GridSearchCV sobre TODOS los datos
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=kf,
    scoring='accuracy',   # o 'f1', 'roc_auc'...
    n_jobs=-1,
    verbose=2,
)

# 6. Un solo fit sobre X, y completos
grid.fit(X, y)

# 7. Resultados
print("Mejor arquitectura:", grid.best_params_['mlp__hidden_layer_sizes'])
print("Mejor CV score:", grid.best_score_)
print("Mejor modelo:", grid.best_estimator_)